In [11]:
#2.1
def markov_probabilities(sequence, vocab):
    """
    计算一阶马尔可夫模型的转移概率（加1平滑）
    
    Args:
        sequence: 字符序列，如 "ababc"
        vocab: 词汇表列表，如 ['a', 'b', 'c']
    
    Returns:
        transition_probs: 转移概率字典
    """
    # 统计转移次数
    transition_counts = {}
    for prev in vocab:
        for curr in vocab:
            transition_counts[(prev, curr)] = 0
    
    # 统计序列中的转移
    for i in range(len(sequence) - 1):
        prev = sequence[i]
        curr = sequence[i + 1]
        if prev in vocab and curr in vocab:
            transition_counts[(prev, curr)] += 1
    
    # 计算加1平滑后的概率
    # P(curr|prev) = (count(prev,curr) + 1) / (count(prev) + |V|)
    transition_probs = {}
    for prev in vocab:
        # 计算 prev 的总出现次数（作为前驱）
        prev_count = sum(1 for i in range(len(sequence) - 1) if sequence[i] == prev)
        for curr in vocab:
            count = transition_counts[(prev, curr)]
            prob = (count + 1) / (prev_count + len(vocab))
            transition_probs[(prev, curr)] = prob
    
    return transition_probs

# 测试
sequence = "ababc"
vocab = ['a', 'b', 'c']

probs = markov_probabilities(sequence, vocab)

print("一阶马尔可夫模型转移概率（加1平滑）:")
print(f"1. p(a'|b') = {probs[('b', 'a')]:.4f}")
print(f"2. p(c'|b') = {probs[('b', 'c')]:.4f}")

# 显示所有转移概率
print("\n所有转移概率:")
for (prev, curr), prob in probs.items():
    print(f"  p({curr}'|{prev}') = {prob:.4f}")

一阶马尔可夫模型转移概率（加1平滑）:
1. p(a'|b') = 0.4000
2. p(c'|b') = 0.4000

所有转移概率:
  p(a'|a') = 0.2000
  p(b'|a') = 0.6000
  p(c'|a') = 0.2000
  p(a'|b') = 0.4000
  p(b'|b') = 0.2000
  p(c'|b') = 0.4000
  p(a'|c') = 0.3333
  p(b'|c') = 0.3333
  p(c'|c') = 0.3333


In [12]:
#2.2
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理函数
    
    Args:
        text: 输入文本字符串
        n: 滑动窗口大小（特征序列长度）
    
    Returns:
        vocab: 词汇表字典 {词: ID}
        features: 特征序列列表
        labels: 标签列表
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    tokens = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(tokens)
    # 按频率降序排序，相同频率按字母顺序
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成特征序列和标签
    features = []
    labels = []
    
    for i in range(len(tokens) - n):
        features.append(tokens[i:i+n])
        labels.append(tokens[i+n])
    
    return vocab, (features, labels)

# 测试
if __name__ == "__main__":
    text = "The time machine"
    vocab, (features, labels) = preprocess_text(text, 2)
    print("词汇表:", vocab)
    print("特征:", features)
    print("标签:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time']]
标签: ['machine']


In [13]:
#3.1
def numerical_gradient_check():
    """
    数值验证RNN梯度公式
    """
    np.random.seed(42)
    input_size = 2
    hidden_size = 3
    seq_len = 4
    
    # 初始化权重
    W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
    W_hx = np.random.randn(hidden_size, input_size) * 0.1
    
    # 生成序列
    x_seq = [np.random.randn(input_size) for _ in range(seq_len)]
    h_prev = np.random.randn(hidden_size)
    y_seq = [np.random.randn(hidden_size) for _ in range(seq_len)]
    
    def forward(W_hh_val):
        """前向传播计算损失"""
        h = h_prev.copy()
        loss = 0
        for t in range(seq_len):
            h = np.tanh(np.dot(W_hh_val, h) + np.dot(W_hx, x_seq[t]))
            loss += 0.5 * np.sum((h - y_seq[t]) ** 2)
        return loss
    
    # 解析梯度 (通过BPTT展开)
    # 这里简化为只展示数值验证方法
    epsilon = 1e-6
    grad_analytic = np.zeros_like(W_hh)
    
    for i in range(W_hh.shape[0]):
        for j in range(W_hh.shape[1]):
            W_hh_plus = W_hh.copy()
            W_hh_plus[i, j] += epsilon
            W_hh_minus = W_hh.copy()
            W_hh_minus[i, j] -= epsilon
            
            grad_numeric = (forward(W_hh_plus) - forward(W_hh_minus)) / (2 * epsilon)
            grad_analytic[i, j] = grad_numeric
    
    print("数值梯度验证:")
    print("梯度矩阵形状:", grad_analytic.shape)
    print("梯度矩阵（部分）:\n", grad_analytic[:2, :2])
    
    # 说明梯度消失/爆炸条件
    print("\n梯度消失/爆炸条件:")
    print("定义: 在RNN中，梯度通过时间步反向传播时，会乘以W_hh的幂次")
    print("若 ||W_hh|| < 1，梯度消失")
    print("若 ||W_hh|| > 1，梯度爆炸")
    print(f"当前 W_hh 的谱范数: {np.linalg.norm(W_hh, ord=2):.4f}")
    print(f"当前 W_hh 的特征值: {np.linalg.eigvals(W_hh)}")

# 运行验证
numerical_gradient_check()

数值梯度验证:
梯度矩阵形状: (3, 3)
梯度矩阵（部分）:
 [[-1.48650344 -0.46866564]
 [ 1.08562033  0.21272456]]

梯度消失/爆炸条件:
定义: 在RNN中，梯度通过时间步反向传播时，会乘以W_hh的幂次
若 ||W_hh|| < 1，梯度消失
若 ||W_hh|| > 1，梯度爆炸
当前 W_hh 的谱范数: 0.2312
当前 W_hh 的特征值: [ 0.12279569+0.j         -0.07174352+0.06124543j -0.07174352-0.06124543j]


In [14]:
#3.2
import numpy as np

class RNNCell:
    def __init__(self, input_size, hidden_size):
        """初始化RNN单元权重"""
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # 初始化权重 (使用小随机数)
        self.W_hx = np.random.randn(hidden_size, input_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.b_h = np.zeros((hidden_size, 1))
        
    def forward(self, x_t, h_prev):
        """
        前向传播
        
        Args:
            x_t: 当前输入 (batch_size, input_size)
            h_prev: 上一隐藏状态 (batch_size, hidden_size)
        
        Returns:
            h_t: 当前隐藏状态 (batch_size, hidden_size)
        """
        # 确保是2D数组
        if x_t.ndim == 1:
            x_t = x_t.reshape(-1, 1)
        if h_prev.ndim == 1:
            h_prev = h_prev.reshape(-1, 1)
        
        batch_size = x_t.shape[0]
        
        # 计算 h_t = tanh(W_hh * h_prev + W_hx * x_t + b_h)
        self.x_t = x_t
        self.h_prev = h_prev
        
        # 线性组合
        self.linear = np.dot(self.W_hh, h_prev.T).T + np.dot(self.W_hx, x_t.T).T + self.b_h.T
        # tanh激活
        self.h_t = np.tanh(self.linear)
        
        return self.h_t
    
    def backward(self, dh_next):
        """
        反向传播（单步）
        
        Args:
            dh_next: 损失对h_t的梯度 (batch_size, hidden_size)
        
        Returns:
            dx_t: 损失对x_t的梯度
            dh_prev: 损失对h_prev的梯度
            dW_hx: 损失对W_hx的梯度
            dW_hh: 损失对W_hh的梯度
            db_h: 损失对b_h的梯度
        """
        batch_size = dh_next.shape[0]
        
        # 1. tanh的导数: d(tanh(z))/dz = 1 - tanh(z)^2
        dtanh = dh_next * (1 - self.h_t ** 2)
        
        # 2. 对线性部分的梯度
        dlinear = dtanh
        
        # 3. 对权重的梯度
        # dW_hh = dlinear * h_prev^T
        dW_hh = np.dot(dlinear.T, self.h_prev) / batch_size
        
        # dW_hx = dlinear * x_t^T
        dW_hx = np.dot(dlinear.T, self.x_t) / batch_size
        
        # db_h = sum of dlinear over batch
        db_h = np.sum(dlinear, axis=0, keepdims=True).T / batch_size
        
        # 4. 对输入的梯度
        # dx_t = W_hx^T * dlinear
        dx_t = np.dot(self.W_hx.T, dlinear.T).T
        
        # dh_prev = W_hh^T * dlinear
        dh_prev = np.dot(self.W_hh.T, dlinear.T).T
        
        return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试
if __name__ == "__main__":
    np.random.seed(42)
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    rnn = RNNCell(input_size, hidden_size)
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    
    # 前向传播
    h_t = rnn.forward(x_t, h_prev)
    print("h_t形状:", h_t.shape)
    print("h_t:\n", h_t)
    
    # 反向传播
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn.backward(dh_next)
    
    print("\n梯度形状:")
    print("dx_t:", dx_t.shape)
    print("dh_prev:", dh_prev.shape)
    print("dW_hx:", dW_hx.shape)
    print("dW_hh:", dW_hh.shape)
    print("db_h:", db_h.shape)

h_t形状: (2, 4)
h_t:
 [[ 0.02628128  0.00655633  0.03396127 -0.01470211]
 [-0.01831118  0.03564895  0.012259    0.0146302 ]]

梯度形状:
dx_t: (2, 3)
dh_prev: (2, 4)
dW_hx: (4, 3)
dW_hh: (4, 4)
db_h: (4, 1)


In [15]:
#4.1
def count_deep_bidirectional_rnn_params(L, H, D, O):
    """
    计算深度双向RNN的参数总数
    
    Args:
        L: 层数
        H: 每层隐藏单元数
        D: 输入维度
        O: 输出维度
    
    Returns:
        total_params: 参数总数
    """
    # 对于双向RNN，每层有前向和后向两个RNN
    
    # 第一层：输入维度 D -> 隐藏维度 H（前向和后向各一个）
    # 每个RNN有 W_hx: H*D, W_hh: H*H, b_h: H
    # 两个方向: 2 * (H*D + H*H + H)
    first_layer = 2 * (H * D + H * H + H)
    
    # 中间层：隐藏维度 H -> 隐藏维度 H（前向和后向各一个）
    # 每个RNN有 W_hx: H*H, W_hh: H*H, b_h: H
    # 两个方向: 2 * (H*H + H*H + H)
    middle_layer = 2 * (H * H + H * H + H)
    
    # 总参数
    if L == 1:
        total = first_layer
    else:
        total = first_layer + (L - 1) * middle_layer
    
    return total

# 测试不同配置
configs = [
    (1, 64, 128, 10),
    (2, 64, 128, 10),
    (3, 128, 256, 20),
    (4, 256, 512, 30),
]

print("深度双向RNN参数计算:")
print("=" * 60)
print(f"{'L':<4} {'H':<6} {'D':<6} {'O':<6} {'参数总数':<15}")
print("-" * 60)

for L, H, D, O in configs:
    params = count_deep_bidirectional_rnn_params(L, H, D, O)
    print(f"{L:<4} {H:<6} {D:<6} {O:<6} {params:<15,}")

深度双向RNN参数计算:
L    H      D      O      参数总数           
------------------------------------------------------------
1    64     128    10     24,704         
2    64     128    10     41,216         
3    128    256    20     230,144        
4    256    512    30     1,181,696      


In [16]:
#4.2
import numpy as np

class BidirectionalRNNEncoder:
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        """
        双向RNN编码器
        
        Args:
            input_dim: 输入维度
            hidden_dim: 隐藏维度
            num_layers: RNN层数
        """
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 简化实现：单层双向RNN
        # 前向RNN
        self.W_hx_f = np.random.randn(hidden_dim, input_dim) * 0.01
        self.W_hh_f = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b_f = np.zeros((hidden_dim, 1))
        
        # 后向RNN
        self.W_hx_b = np.random.randn(hidden_dim, input_dim) * 0.01
        self.W_hh_b = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b_b = np.zeros((hidden_dim, 1))
        
    def rnn_step(self, x_t, h_prev, W_hx, W_hh, b):
        """单步RNN"""
        if x_t.ndim == 1:
            x_t = x_t.reshape(-1, 1)
        if h_prev.ndim == 1:
            h_prev = h_prev.reshape(-1, 1)
        
        linear = np.dot(W_hh, h_prev.T).T + np.dot(W_hx, x_t.T).T + b.T
        h_t = np.tanh(linear)
        return h_t
    
    def forward(self, X):
        """
        前向传播
        
        Args:
            X: 输入序列 (seq_len, batch, input_dim)
        
        Returns:
            outputs: 每个时间步拼接的隐藏状态 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步拼接的隐藏状态 (batch, 2*hidden_dim)
        """
        seq_len, batch_size, _ = X.shape
        
        # 初始化隐藏状态
        h_f = np.zeros((batch_size, self.hidden_dim))
        h_b = np.zeros((batch_size, self.hidden_dim))
        
        # 存储所有时间步的隐藏状态
        hidden_f = []
        hidden_b = []
        
        # 前向传播（从左到右）
        for t in range(seq_len):
            x_t = X[t]
            h_f = self.rnn_step(x_t, h_f, self.W_hx_f, self.W_hh_f, self.b_f)
            hidden_f.append(h_f.copy())
        
        # 后向传播（从右到左）
        for t in range(seq_len - 1, -1, -1):
            x_t = X[t]
            h_b = self.rnn_step(x_t, h_b, self.W_hx_b, self.W_hh_b, self.b_b)
            hidden_b.append(h_b.copy())
        hidden_b = hidden_b[::-1]  # 反转回原始顺序
        
        # 拼接前向和后向隐藏状态
        outputs = []
        for t in range(seq_len):
            concat = np.concatenate([hidden_f[t], hidden_b[t]], axis=1)
            outputs.append(concat)
        
        outputs = np.array(outputs)  # (seq_len, batch, 2*hidden_dim)
        
        # 最终时间步的拼接隐藏状态
        final_state = outputs[-1]  # 取最后一个时间步
        
        return outputs, final_state

# 测试
if __name__ == "__main__":
    np.random.seed(42)
    seq_len = 4
    batch_size = 2
    input_dim = 3
    hidden_dim = 5
    
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
    X = np.random.randn(seq_len, batch_size, input_dim)
    
    outputs, final_state = encoder.forward(X)
    print("输出形状 (seq_len, batch, 2*hidden_dim):", outputs.shape)
    print("最终状态形状 (batch, 2*hidden_dim):", final_state.shape)
    print("\n输出:\n", outputs)

输出形状 (seq_len, batch, 2*hidden_dim): (4, 2, 10)
最终状态形状 (batch, 2*hidden_dim): (2, 10)

输出:
 [[[ 0.00798708 -0.00764201 -0.00766666 -0.00972946 -0.03284473
   -0.00274124 -0.01518423  0.00996453 -0.00063773  0.01915509]
  [-0.00470625 -0.00482546 -0.01203305  0.0032716   0.02286564
   -0.00450122  0.01731184 -0.00790498  0.0079303  -0.0067456 ]]

 [[ 0.0012218   0.0148579   0.0196859   0.00555448  0.00516422
    0.00784402 -0.00394756 -0.00260383 -0.01266778 -0.00964872]
  [ 0.00837374  0.00508461  0.00422179 -0.00208232 -0.01703739
    0.00294878 -0.01021288  0.0019549  -0.01250774  0.00671825]]

 [[-0.0057926  -0.00911652 -0.01190422 -0.00021552  0.01090023
   -0.0055137   0.00964529 -0.00152346  0.01353445 -0.00092672]
  [-0.00581381 -0.02335361 -0.02203268 -0.01060303 -0.01388909
   -0.01075748 -0.00198426  0.01076017  0.02627254  0.01456778]]

 [[-0.0088817   0.00381244  0.00494849  0.00766605  0.02921563
    0.0012726   0.01364064 -0.00736677  0.0045999  -0.01606061]
  [-0.0063193

In [17]:
#5.1
def skipgram_negative_sampling_loss(v_c, u_o, negative_samples, K):
    """
    计算Skip-gram负采样的损失函数（演示）
    
    Args:
        v_c: 中心词向量
        u_o: 上下文词向量
        negative_samples: 负样本词向量列表 [u_n1, u_n2, ...]
        K: 负样本数量
    
    Returns:
        loss: 损失值
    """
    import numpy as np
    
    # 正样本部分: -log(sigmoid(u_o^T * v_c))
    pos_term = -np.log(1 / (1 + np.exp(-np.dot(u_o, v_c))))
    
    # 负样本部分: -sum_{k=1}^K log(sigmoid(-u_nk^T * v_c))
    neg_sum = 0
    for u_n in negative_samples[:K]:
        neg_sum += np.log(1 / (1 + np.exp(np.dot(u_n, v_c))))
    neg_term = -neg_sum
    
    loss = pos_term + neg_term
    
    print("Skip-gram负采样损失函数演示:")
    print(f"正样本项: {pos_term:.4f}")
    print(f"负样本项: {neg_term:.4f}")
    print(f"总损失: {loss:.4f}")
    
    return loss

# 测试
np.random.seed(42)
d = 4
v_c = np.random.randn(d)
u_o = np.random.randn(d)
negative_samples = [np.random.randn(d) for _ in range(5)]

loss = skipgram_negative_sampling_loss(v_c, u_o, negative_samples, K=3)

Skip-gram负采样损失函数演示:
正样本项: 0.1147
负样本项: 0.4598
总损失: 0.5744


In [18]:
#5.2
import numpy as np

class CBOW:
    def __init__(self, vocab_size, embedding_dim):
        """
        CBOW模型
        
        Args:
            vocab_size: 词汇表大小 V
            embedding_dim: 嵌入维度 d
        """
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # 输入权重矩阵 W (V, d)
        self.W = np.random.randn(vocab_size, embedding_dim) * 0.01
        # 输出权重矩阵 W_out (d, V)
        self.W_out = np.random.randn(embedding_dim, vocab_size) * 0.01
        
    def forward(self, context_indices, target_index):
        """
        前向传播和损失计算
        
        Args:
            context_indices: 上下文词索引列表，每个样本为 (context_size,) 或 (batch, context_size)
            target_index: 目标中心词索引，标量或 (batch,)
        
        Returns:
            loss: 交叉熵损失值
            probs: 输出概率分布
        """
        # 处理输入格式
        if isinstance(context_indices[0], (list, np.ndarray)):
            # 批量输入
            batch_size = len(context_indices)
            context_size = len(context_indices[0])
        else:
            # 单样本输入
            batch_size = 1
            context_size = len(context_indices)
            context_indices = [context_indices]
            target_index = [target_index]
        
        # 1. 获取上下文词的嵌入向量并求平均
        context_embeddings = []
        for sample in context_indices:
            # 获取每个上下文词的嵌入 (context_size, d)
            sample_emb = np.array([self.W[idx] for idx in sample])
            # 求平均 -> (d,)
            avg_emb = np.mean(sample_emb, axis=0)
            context_embeddings.append(avg_emb)
        
        # (batch, d)
        hidden = np.array(context_embeddings)
        
        # 2. 计算输出分数 (batch, V)
        scores = np.dot(hidden, self.W_out)
        
        # 3. Softmax得到概率分布 (batch, V)
        exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
        probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        
        # 4. 计算交叉熵损失
        losses = []
        for i, target in enumerate(target_index):
            # 防止log(0)
            loss = -np.log(probs[i, target] + 1e-8)
            losses.append(loss)
        
        loss = np.mean(losses)
        
        return loss, probs

# 测试
if __name__ == "__main__":
    np.random.seed(42)
    vocab_size = 10
    embedding_dim = 4
    context_size = 2
    
    cbow = CBOW(vocab_size, embedding_dim)
    
    # 单样本测试
    context = [1, 3]  # 上下文词索引
    target = 5        # 中心词索引
    
    loss, probs = cbow.forward(context, target)
    print("单样本测试:")
    print("Loss:", loss)
    print("概率分布形状:", probs.shape)
    print("概率分布前5个:", probs[0][:5])
    
    # 批量测试
    batch_context = [[1, 3], [2, 4], [5, 6]]
    batch_target = [5, 7, 8]
    
    loss_batch, probs_batch = cbow.forward(batch_context, batch_target)
    print("\n批量测试:")
    print("Loss:", loss_batch)
    print("概率分布形状:", probs_batch.shape)

单样本测试:
Loss: 2.3026648267060414
概率分布形状: (1, 10)
概率分布前5个: [0.09999946 0.10000804 0.10001022 0.09999809 0.09998778]

批量测试:
Loss: 2.302666989126499
概率分布形状: (3, 10)


In [19]:
#6.1
def compute_scaled_dot_product_attention(Q, K, V):
    """
    计算缩放点积注意力（数值示例）
    
    Args:
        Q: (2, 4)
        K: (3, 4)
        V: (3, 5)
    
    Returns:
        output: (2, 5)
        scores: (2, 3)
        attention_weights: (2, 3)
    """
    import numpy as np
    
    d_k = Q.shape[1]  # 4
    
    # 1. 计算得分矩阵
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    
    # 2. Softmax
    exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
    attention_weights = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    
    # 3. 加权求和
    output = np.dot(attention_weights, V)
    
    print("=== 缩放点积注意力计算 ===")
    print(f"Q 形状: {Q.shape}")
    print(f"K 形状: {K.shape}")
    print(f"V 形状: {V.shape}")
    print(f"\n1. 得分矩阵 (QK^T / sqrt(d_k)):")
    print(scores)
    print(f"\n2. Attention权重 (Softmax):")
    print(attention_weights)
    print(f"\n3. 输出 (加权求和):")
    print(output)
    print(f"\n输出形状: {output.shape}")
    
    return output, scores, attention_weights

# 测试
np.random.seed(42)
Q = np.random.randn(2, 4)
K = np.random.randn(3, 4)
V = np.random.randn(3, 5)

output, scores, weights = compute_scaled_dot_product_attention(Q, K, V)

=== 缩放点积注意力计算 ===
Q 形状: (2, 4)
K 形状: (3, 4)
V 形状: (3, 5)

1. 得分矩阵 (QK^T / sqrt(d_k)):
[[-0.65884095 -0.79443288 -1.64281711]
 [-0.55317835 -1.382109   -1.17711663]]

2. Attention权重 (Softmax):
[[0.44503374 0.38860296 0.1663633 ]
 [0.50701047 0.22131809 0.27167143]]

3. 输出 (加权求和):
[[ 0.5952661  -0.23960648  0.17380425 -1.04343526 -0.21878045]
 [ 0.60418195  0.13400442  0.11371947 -1.1426443  -0.11710289]]

输出形状: (2, 5)


In [20]:
#6.2
import numpy as np

class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        """
        多头注意力机制
        
        Args:
            d_model: 模型维度
            num_heads: 头数
        """
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影矩阵
        self.W_Q = np.random.randn(d_model, d_model) * 0.01
        self.W_K = np.random.randn(d_model, d_model) * 0.01
        self.W_V = np.random.randn(d_model, d_model) * 0.01
        self.W_O = np.random.randn(d_model, d_model) * 0.01
        
    def scaled_dot_product_attention(self, Q, K, V):
        """
        缩放点积注意力
        
        Args:
            Q: (seq_len, batch, d_k)
            K: (seq_len, batch, d_k)
            V: (seq_len, batch, d_v)
        
        Returns:
            output: (seq_len, batch, d_v)
            attention_weights: (seq_len, seq_len, batch)
        """
        # 转置以便计算: (batch, seq_len, d_k)
        Q = Q.transpose(1, 0, 2)
        K = K.transpose(1, 0, 2)
        V = V.transpose(1, 0, 2)
        
        # 计算得分: (batch, seq_len, seq_len)
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(self.d_k)
        
        # Softmax: (batch, seq_len, seq_len)
        exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
        
        # 加权求和: (batch, seq_len, d_v)
        output = np.matmul(attention_weights, V)
        
        # 转回: (seq_len, batch, d_v)
        output = output.transpose(1, 0, 2)
        attention_weights = attention_weights.transpose(1, 2, 0)
        
        return output, attention_weights
    
    def forward(self, X):
        """
        前向传播
        
        Args:
            X: (seq_len, batch, d_model)
        
        Returns:
            output: (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影得到Q, K, V
        Q = np.dot(X.reshape(-1, self.d_model), self.W_Q).reshape(seq_len, batch_size, self.d_model)
        K = np.dot(X.reshape(-1, self.d_model), self.W_K).reshape(seq_len, batch_size, self.d_model)
        V = np.dot(X.reshape(-1, self.d_model), self.W_V).reshape(seq_len, batch_size, self.d_model)
        
        # 2. 分割成多头
        # (seq_len, batch, num_heads, d_k)
        Q_heads = Q.reshape(seq_len, batch_size, self.num_heads, self.d_k)
        K_heads = K.reshape(seq_len, batch_size, self.num_heads, self.d_k)
        V_heads = V.reshape(seq_len, batch_size, self.num_heads, self.d_v)
        
        # 交换维度: (num_heads, seq_len, batch, d_k)
        Q_heads = Q_heads.transpose(2, 0, 1, 3)
        K_heads = K_heads.transpose(2, 0, 1, 3)
        V_heads = V_heads.transpose(2, 0, 1, 3)
        
        # 3. 每个头计算注意力
        head_outputs = []
        for i in range(self.num_heads):
            Q_i = Q_heads[i]  # (seq_len, batch, d_k)
            K_i = K_heads[i]  # (seq_len, batch, d_k)
            V_i = V_heads[i]  # (seq_len, batch, d_v)
            
            output_i, _ = self.scaled_dot_product_attention(Q_i, K_i, V_i)
            head_outputs.append(output_i)
        
        # 4. 拼接所有头的输出
        # (num_heads, seq_len, batch, d_v) -> (seq_len, batch, num_heads * d_v)
        concat_output = np.stack(head_outputs, axis=0)  # (num_heads, seq_len, batch, d_v)
        concat_output = concat_output.transpose(1, 2, 0, 3)  # (seq_len, batch, num_heads, d_v)
        concat_output = concat_output.reshape(seq_len, batch_size, self.d_model)
        
        # 5. 最终线性层
        output = np.dot(concat_output.reshape(-1, self.d_model), self.W_O).reshape(seq_len, batch_size, self.d_model)
        
        return output

# 测试
if __name__ == "__main__":
    np.random.seed(42)
    d_model = 4
    num_heads = 2
    seq_len = 3
    batch_size = 2
    
    mha = MultiHeadAttention(d_model, num_heads)
    X = np.random.randn(seq_len, batch_size, d_model)
    
    output = mha.forward(X)
    print("输入形状:", X.shape)
    print("输出形状:", output.shape)
    print("\n输出:\n", output)

输入形状: (3, 2, 4)
输出形状: (3, 2, 4)

输出:
 [[[ 2.54059795e-04 -5.61333266e-06 -4.15048786e-04 -4.35745609e-04]
  [-1.05981946e-04  6.36457532e-06  1.62138192e-04  1.63137002e-04]]

 [[ 2.54061746e-04 -5.59282251e-06 -4.15087265e-04 -4.35777991e-04]
  [-1.05972581e-04  6.33820975e-06  1.62152993e-04  1.63138708e-04]]

 [[ 2.54051313e-04 -5.67126878e-06 -4.15000003e-04 -4.35734444e-04]
  [-1.05971493e-04  6.37763490e-06  1.62198414e-04  1.63220730e-04]]]
